# 38a · Fine-tune RAD-DINO — **chỉ ảnh**, không EHR

Đo **một** thứ: backbone được học đáng bao nhiêu. Cohort cũ đo `LP 0.3191 → FT 0.4201 = +0.101` —
lớn hơn tất cả những gì `35` và `37` đo được cộng lại.

**`38b` sẽ đóng băng backbone đã fine-tune này rồi chạy lại toàn bộ arm EHR.** Lúc đó máy móc của
`37` dùng lại được — một forward, nhiều arm, mỗi arm ~15% chi phí — nên bảng xếp hạng arm EHR sẽ
được đo **trên cấu hình đi vào bài**, kèm 2–3 seed. Đó là lý do `38a` **không** có arm EHR nào.

```
38a   tran cua truc ANH
38b   dong bang backbone da FT + arm EHR   -> "anh da toi da roi, EHR CON them gi"  (phep thu KHO NHAT)
39    fine-tune CO EHR trong vong lap      -> "EHR co dinh hinh duoc BIEU DIEN khong"  (B2d)
```

---

## Hai arm, hai backbone, **cùng một vòng lặp**

| arm | backbone | đầu đọc | vai trò |
|---|---|---|---|
| `lp` | **đông cứng** (RAD-DINO pretrain) | ML-Decoder | **mốc nội bộ** |
| `ft` | **học** | ML-Decoder | `ft − lp` = đóng góp của fine-tune |

Cùng thứ tự dữ liệu, cùng phép tăng cường, cùng lịch LR ⇒ `ft − lp` là **hiệu ghép cặp**, phương sai
nhỏ hơn hẳn so với lấy `0.2405` của `37` làm mốc.

> Bài học `mean_lin` của `37`: ảnh ở đây qua tăng cường + mã hoá lại JPEG, lịch LR khác, nên
> **không được** lấy số của notebook khác làm mốc. Mốc phải đo **trong** notebook này.

**Đầu đọc khởi tạo từ checkpoint `37`.** Đó chính là **pha LP của LP-FT** — miễn phí vì ta đã có.
Kumar et al. (2022): LP-FT giữ đặc trưng pretrain tốt hơn FT thẳng từ đầu đọc ngẫu nhiên.

**LR của backbone = LR đầu đọc / 10.** Đầu đọc đã ở chỗ tốt (khởi tạo từ `37`), backbone chỉ cần
dịch nhẹ. Cùng LR thì backbone bị đầu đọc kéo đi và quên đặc trưng pretrain — đúng rủi ro mà
"mở một phần" định tránh, nhưng giải bằng LR thì **không cắt mất trần**.
Dự phòng có điều kiện rõ ràng: nếu đường cong **tail** tụt sớm thì mới rơi về mở 4 block cuối.

---

## Kỷ luật kế thừa

`asl40` · **không early-stop, lấy epoch cuối**, cosine trải đúng trên `E` · clip **từng arm** ·
ngưỡng đọc từ biên độ seed đo được, **không** dùng `0.005` cho tail.

**Và một kỷ luật riêng cho `38`:** đây là chỗ dễ overfit val nhất cả dự án — mỗi cấu hình tốn hàng giờ
nên sẽ có cám dỗ thử nhiều rồi lấy cái tốt nhất trên val. `E` chọn trên **lát cắt 15% của train**,
mọi lần chạy ghi vào `_runs.jsonl`, **test chưa chạm**.

**Ba con số ĐO chứ không đoán:** `img/s` thật (C5 dò batch), `E` tối ưu (đường cong theo tier),
ngưỡng đọc (biên độ seed, từ `38b`).

## C0 · Config

In [ ]:
import os, json, math, time, random, shutil, tarfile, io as _io, re, glob
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = './planB_out'

OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024')
EMB_DIR = os.path.join(OUT_DIR, 'embeddings')
RUN_TAG = 'a4_E6_lr5e4'                        # doi tag => thu muc moi, lan chay cu con nguyen
RES38   = os.path.join(OUT_DIR, 'stage4_nb38a', RUN_TAG); os.makedirs(RES38, exist_ok=True)
CKPT    = os.path.join(RES38, 'ckpt');           os.makedirs(CKPT, exist_ok=True)
LOCAL   = '/content/cxrlt_518'                  # DUNG CHUNG voi 37 — khong stage lai neu con

W_FRONTAL, W_LATERAL = 0.7, 0.3
SEED    = 42
SIZE    = 518
EPOCHS  = 6                     # `E` va `LR` doi chac qua TONG QUANG DUONG:
                                #   Sigma_lr = 0.5 * LR_dinh * so_buoc  (cosine)
                                #   a2  E=3 LR 1e-3  2,748 buoc  -> 1.374
                                #   a4  E=6 LR 5e-4  5,496 buoc  -> 1.374   BANG NHAU
                                # => giu quang duong CO DINH, chi doi SO BUOC vs DO LON BUOC.
                                # Gap doi so buoc, va moi anh thay 6 luot tang cuong thay vi 3.
                                # (E=2 cho Sigma_lr 0.92 — HUT so voi toi uu quan sat ~1.3.)
# (ghi chu cu) a1 E=8: ft dinh ep3 (0.2395 CHUA u) roi sut ve 0.2231.
                                # a2 E=3: diem u xong 0.2412 > dinh chua u cua a1 => u LR co gia tri.
                                #         Nhung dinh lai dich ve ep2 (0.2458, tail 0.0988).
                                # => E=2 dat diem u xong DUNG vao ep2.
# (ghi chu cu) LAN DO DAU. `04b` dinh o ep5/12 nhung do la ViT tren 13 nhan
                                # cohort cu, va `35` da chung minh "dinh" phu thuoc LICH chu khong
                                # phai hoi tu. C8 canh bao neu duong cong con dang len.
BATCH, ACCUM = 120, 2           # EFF = 240, GIU NGUYEN qua ca a1/a2/a3 => `E` la bien DUY NHAT.
                                # (144x2 = 288 se doi ca so buoc lan batch hieu dung.)
LR_HEAD, LR_MULT_BB = 5e-4, 0.1 # backbone = 1/10 LR dau doc (LP-FT). 5e-4 de Sigma_lr = a2.
WD, WARMUP, CLIP = 1e-2, 0.05, 5.0
NW = min(8, (os.cpu_count() or 2))

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
assert dev == 'cuda', 'KHONG thay GPU -> Runtime > Change runtime type > GPU roi chay lai tu C0.'
assert torch.cuda.is_bf16_supported(), (
    'GPU khong ho tro bf16. Fine-tune fp16 can GradScaler — sua C7 truoc khi chay.')
AMP = torch.bfloat16
_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'thiet bi: {torch.cuda.get_device_name(0)} | VRAM {_vram:.0f} GB | bf16 OK')

INV   = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']; C = len(LAB40)
TIER  = {t: np.array([LAB40.index(l) for l in INV['tier'][t]]) for t in ('head', 'medium', 'tail')}

ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
Y = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id'] + LAB40)
       .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())
assert (Y.dicom_id.to_numpy() == ORDER.dicom_id.to_numpy()).all()
YM = Y[LAB40].to_numpy('float32')

P_VIEW = os.path.join(OUT_DIR, 'cxrlt_view_resolved.parquet')
V = ((pd.read_parquet(P_VIEW)[['dicom_id', 'view_group', 'use']]
        .set_index('dicom_id').loc[ORDER.dicom_id].reset_index()) if os.path.exists(P_VIEW)
     else ORDER[['dicom_id', 'view_group']].assign(use=lambda d: d.view_group != 'unknown'))

SPLIT = ORDER.split.to_numpy(); SUB = ORDER.subject_id.to_numpy()
TR, VA = SPLIT == 'train', SPLIT == 'val'
_rs = np.random.default_rng(SEED); _subs = np.unique(SUB[TR])
_sel = _rs.choice(_subs, size=int(round(0.15 * len(_subs))), replace=False)
SL = TR & np.isin(SUB, _sel); TF_ = TR & ~SL          # CUNG cach cat nhu 35/37
assert not (set(SUB[SL]) & set(SUB[TF_])) and not (set(SUB[TR]) & set(SUB[VA]))
I_TF, I_SL, I_VA = np.where(TF_)[0], np.where(SL)[0], np.where(VA)[0]

# --- so lan cham VAL: 38 la cho de overfit val nhat ca du an ---
P_RUNS = os.path.join(OUT_DIR, 'stage4_nb38a', '_runs.jsonl')
print(f'{len(ORDER):,} anh | khop {len(I_TF):,} | chon {len(I_SL):,} | val {len(I_VA):,}')
print(f'ghi vao: {RES38}')
print(f'E={EPOCHS} | LR dau doc {LR_HEAD} | LR backbone {LR_HEAD*LR_MULT_BB} | NW {NW}')
_n_prev = sum(1 for _ in open(P_RUNS)) if os.path.exists(P_RUNS) else 0
print(f'so lan chay 38a da ghi truoc day: {_n_prev}  <- moi lan cham val deu tinh')

## C1 · Ảnh local — dùng chung với `37`

`37` đã stage `377,110` ảnh `518` vào `/content/cxrlt_518`. Nếu runtime chưa chết thì cell này chạy
tức thì. Nếu đã mất, nó stage lại (~20–30 phút, 8 luồng).

In [ ]:
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

os.makedirs(LOCAL, exist_ok=True)
_have = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
print(f'anh local hien co: {_have:,} / {len(ORDER):,}')

if _have < len(ORDER) * 0.999:
    SHARD_DIR = os.path.join(DATA_DIR, 'cxr_1022_letterbox_cxrlt')   # tu 28c C0
    assert os.path.isdir(SHARD_DIR), f'khong thay {SHARD_DIR}'
    TARS = sorted(os.path.join(SHARD_DIR, f) for f in os.listdir(SHARD_DIR)
                  if f.endswith('.tar') and f.startswith('lb_'))
    DONE = os.path.join(LOCAL, '_done.json')
    done = set(json.load(open(DONE))) if os.path.exists(DONE) else set()
    todo = [t for t in TARS if os.path.basename(t) not in done]
    print(f'stage lai {len(todo)}/{len(TARS)} tar')

    def _one(pl):
        name, blob = pl
        dst = os.path.join(LOCAL, os.path.basename(name))
        if os.path.exists(dst): return 0
        Image.open(_io.BytesIO(blob)).convert('L').resize((SIZE, SIZE), Image.BICUBIC) \
             .save(dst, 'JPEG', quality=95)
        return 1
    t0 = time.time()
    with ThreadPoolExecutor(8) as pool:
        for tp in tqdm(todo, desc='stage', unit='tar'):
            with open(tp, 'rb') as f: buf = f.read()      # doc BULK; lap tren mount FUSE se TREO
            with tarfile.open(fileobj=_io.BytesIO(buf)) as tar:
                jobs = [(m.name, tar.extractfile(m).read())
                        for m in tar.getmembers() if m.name.endswith('.jpg')]
            list(pool.map(_one, jobs)); del buf, jobs
            done.add(os.path.basename(tp)); json.dump(sorted(done), open(DONE, 'w'))
    _have = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
    print(f'xong: {_have:,} anh | {(time.time()-t0)/60:.1f} phut')
assert _have >= len(ORDER) * 0.999, f'thieu {len(ORDER)-_have:,} anh'

## C2 · **Hai** backbone + `Dataset`

`bb_lp` đông cứng (RAD-DINO pretrain nguyên bản), `bb_ft` học. Hai bản sao độc lập, cùng khởi tạo.
Cùng một batch đi qua cả hai ⇒ `ft − lp` là hiệu ghép cặp thật.

Tăng cường **giống `06b`/`37`**: `RandomResizedCrop(.85–1.0)` + xoay `7°`. Nó vừa chính đáng hoá việc
tính lại backbone mỗi epoch, vừa là chính quy hoá cần thiết khi backbone có `86M` tham số được học.

In [ ]:
from transformers import AutoModel, AutoImageProcessor
from torchvision import transforms as T

proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
MEAN, STD = proc.image_mean[0], proc.image_std[0]
train_tf = T.Compose([T.RandomResizedCrop(SIZE, scale=(0.85, 1.0), ratio=(0.95, 1.05),
                                          interpolation=T.InterpolationMode.BICUBIC),
                      T.RandomRotation(7), T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])
eval_tf  = T.Compose([T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])

bb_lp = AutoModel.from_pretrained('microsoft/rad-dino').to(dev).eval()
for p in bb_lp.parameters(): p.requires_grad_(False)
bb_ft = AutoModel.from_pretrained('microsoft/rad-dino').to(dev)      # CUNG trong so, se duoc hoc
D_CXR = bb_lp.config.hidden_size
_n_bb = sum(p.numel() for p in bb_ft.parameters())
print(f'backbone: hidden {D_CXR} | patch {bb_lp.config.patch_size} | {_n_bb:,} tham so moi ban')
assert not any(p.requires_grad for p in bb_lp.parameters()), 'bb_lp chua dong cung'
assert all(p.requires_grad for p in bb_ft.parameters()), 'bb_ft khong hoc duoc'

class DS(Dataset):
    def __init__(s, idx, tf): s.idx, s.tf = np.asarray(idx), tf
    def __len__(s): return len(s.idx)
    def __getitem__(s, k):
        i = int(s.idx[k])
        im = Image.open(os.path.join(LOCAL, f'{ORDER.dicom_id.iloc[i]}.jpg')).convert('L')
        return dict(image=s.tf(Image.merge('RGB', (im, im, im))),
                    y=torch.from_numpy(YM[i]), row=i)

def loader(idx, tf, shuffle, bs, nw=NW):
    return DataLoader(DS(idx, tf), batch_size=bs, shuffle=shuffle, num_workers=nw,
                      pin_memory=True, drop_last=shuffle,
                      persistent_workers=nw > 0, prefetch_factor=4 if nw > 0 else None)

def fwd(bb, img, grad):
    ctx = torch.enable_grad() if grad else torch.no_grad()
    with ctx, torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
        h = bb(pixel_values=img).last_hidden_state
    return h[:, 1:].float()                                  # bo CLS, giu 1369 patch token

_b = next(iter(loader(I_VA[:8], eval_tf, False, 8, nw=0)))
_p = fwd(bb_lp, _b['image'].to(dev), False)
N_TOK = _p.shape[1]
print(f'patch {tuple(_p.shape)} <- {N_TOK} token')
assert N_TOK == (SIZE // bb_lp.config.patch_size) ** 2

## C3 · ML-Decoder + ASL — **verbatim `37`**

Không sửa gì, để `38a` so được với `37` về mặt kiến trúc. `query_embed.weight.requires_grad = False`
là thiết kế của bài gốc (Ridnik et al., WACV 2023), đừng "sửa".

In [ ]:
class GroupFC(nn.Module):
    def __init__(s, C, d):
        super().__init__(); s.w = nn.Parameter(torch.randn(C, d) * d ** -0.5); s.b = nn.Parameter(torch.zeros(C))
    def forward(s, h): return (h * s.w).sum(-1) + s.b

class XAttn(nn.Module):
    def __init__(s, d, nhead=8, ff=1024, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n1, s.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        s.ff = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Dropout(drop), nn.Linear(ff, d))
    def forward(s, q, kv, need_w=False):
        a, w = s.at(q, kv, kv, need_weights=need_w, average_attn_weights=True)
        h = s.n1(q + a)
        return s.n2(h + s.ff(h)), w

class MLDecoder(nn.Module):
    '''Q = C query lop (KHONG hoc), K/V = 1369 patch token. Bo self-attention giua cac query
       nen chi phi TUYEN TINH theo so lop: 40 x 1369, khong phai 1369^2 = 1.87M.'''
    def __init__(s, C, d_in, d=512, nhead=8, ff=1024, drop=0.1):
        super().__init__()
        s.q = nn.Embedding(C, d); s.q.weight.requires_grad_(False)
        s.pin = nn.Linear(d_in, d)
        s.img = XAttn(d, nhead, ff, drop)
        s.out = GroupFC(C, d)
    def forward(s, patch, need_w=False):
        q = s.q.weight.unsqueeze(0).expand(patch.shape[0], -1, -1)
        h, w = s.img(q, s.pin(patch), need_w)
        return s.out(h), w

class ASL(nn.Module):
    '''Verbatim nb11/12/13/15. gn=4, gp=0 la cau hinh 35 da chot.'''
    def __init__(s, gn=4, gp=0, clip=0.05, eps=1e-8):
        super().__init__(); s.gn, s.gp, s.clip, s.eps = gn, gp, clip, eps
    def forward(s, x, y):
        y = y.float(); xp = torch.sigmoid(x); xn = 1 - xp
        if s.clip: xn = (xn + s.clip).clamp(max=1)
        lo = y*torch.log(xp.clamp(min=s.eps)) + (1-y)*torch.log(xn.clamp(min=s.eps))
        pt = xp*y + xn*(1-y); g = s.gp*y + s.gn*(1-y)
        return -(lo * torch.pow(1-pt, g)).mean()
print('kien truc san sang')

## C4 · Hai arm + khởi tạo đầu đọc từ `37` + cổng `assert`

In [ ]:
import copy
D_MLD = 512
torch.manual_seed(SEED)
# Dung SAO CHEP SAU, khong tao hai lan: `MLDecoder(...)` goi hai lan lien tiep tieu RNG khac nhau
# nen hai dau doc se khoi tao KHAC nhau, va `ft - lp` se lan ca chenh lech khoi tao.
# (Dry-run bat duoc: lech 4.40 giua hai ban.)
_h0  = MLDecoder(C, D_CXR, D_MLD)
HEAD = nn.ModuleDict({'lp': _h0, 'ft': copy.deepcopy(_h0)}).to(dev)

# --- nap dau doc `mldec` cua 37 => day chinh la pha LP cua LP-FT ---
_cands = sorted(glob.glob(os.path.join(OUT_DIR, 'stage4_nb37', '**', 'ckpt', 'resume.pt'),
                          recursive=True), key=os.path.getmtime, reverse=True)
CK37 = _cands[0] if _cands else None
if CK37:
    _sd = torch.load(CK37, map_location=dev, weights_only=False)['mod']
    _h = {k[len('mldec.'):]: v for k, v in _sd.items() if k.startswith('mldec.')}
    _miss = set(HEAD['lp'].state_dict()) - set(_h)
    if _miss:
        print(f'!! ckpt 37 thieu {len(_miss)} khoa cua dau doc -> KHOI TAO NGAU NHIEN. {sorted(_miss)[:3]}')
        CK37 = None
    else:
        HEAD['lp'].load_state_dict(_h); HEAD['ft'].load_state_dict(_h)
        print(f'nap dau doc `mldec` tu {CK37}')
        print('  => day la pha LP cua LP-FT: dau doc DA o cho tot, backbone chi can dich nhe.')
else:
    print('!! khong thay ckpt 37 -> dau doc khoi tao ngau nhien (van chay duoc, nhung mat pha LP)')

# --- cong ---
assert not any(p.requires_grad for p in bb_lp.parameters()), 'bb_lp phai dong cung'
assert all(p.requires_grad for p in bb_ft.parameters()),      'bb_ft phai hoc'
for t in ('lp', 'ft'):
    assert not HEAD[t].q.weight.requires_grad, f'{t}: query_embed dang hoc (thiet ke bai goc)'
# hai dau doc PHAI khoi dau giong het nhau, neu khong `ft - lp` lan ca chenh lech khoi tao
with torch.no_grad():
    _d = max((HEAD['lp'].state_dict()[k] - HEAD['ft'].state_dict()[k]).abs().max().item()
             for k in HEAD['lp'].state_dict())
assert _d < 1e-6, f'hai dau doc khoi tao KHAC nhau (lech {_d:.2e}) -> ft-lp se lan chenh lech init'
# hai backbone PHAI khoi dau giong het nhau
with torch.no_grad():
    _db = max((a - b).abs().max().item()
              for a, b in zip(bb_lp.state_dict().values(), bb_ft.state_dict().values()))
assert _db < 1e-6, f'hai backbone khoi tao KHAC nhau (lech {_db:.2e})'
_nh = sum(p.numel() for p in HEAD['lp'].parameters() if p.requires_grad)
print(f'\ncong: bb_lp dong cung OK | bb_ft hoc OK | query co dinh OK')
print(f'      hai dau doc khoi tao GIONG HET (lech {_d:.1e}) | hai backbone giong het (lech {_db:.1e})')
print(f'tham so: dau doc {_nh:,}/arm | backbone {_n_bb:,} (chi `ft` hoc)')

## C5 · **Dò batch — đo `img/s` thật**

`04b` đo `150 img/s` (`BATCH 48 @518 bf16`) trên card khác. Bảng A5.2d của kế hoạch từng ngoại suy
theo tỉ lệ GMAC và **đó là ước lượng, không phải số đo** — lần này đo.

Cell này chạy vài bước thật ở mỗi kích thước batch, báo `img/s` và VRAM, rồi **chiếu ra tổng thời
gian** cho `E` epoch. Chọn batch xong thì sửa `BATCH`/`ACCUM` ở `C0` và chạy lại `C0`.

In [ ]:
def probe(sizes=(48, 96, 144, 192, 240), n_step=6):
    crit_ = ASL(gn=4, gp=0).to(dev)
    rows = []
    _cuda = dev == 'cuda'
    for bs in sizes:
        if _cuda: torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        try:
            dl = loader(I_TF[:bs*(n_step+2)], train_tf, True, bs, nw=NW)
            it = iter(dl); opt_ = torch.optim.AdamW(
                [p for p in list(bb_ft.parameters()) + list(HEAD['ft'].parameters())
                 if p.requires_grad], lr=1e-8)
            for _ in range(2): b = next(it)                       # ham nong
            if _cuda: torch.cuda.synchronize()
            t0 = time.time()
            for _ in range(n_step):
                b = next(it); img = b['image'].to(dev, non_blocking=True); y = b['y'].to(dev)
                pl = fwd(bb_lp, img, False)                        # arm lp: khong gradient
                pf = fwd(bb_ft, img, True)                         # arm ft: co gradient
                l = crit_(HEAD['lp'](pl)[0].float(), y) + crit_(HEAD['ft'](pf)[0].float(), y)
                l.backward(); opt_.step(); opt_.zero_grad(set_to_none=True)
            if _cuda: torch.cuda.synchronize()
            ips = n_step * bs / (time.time() - t0)
            gb  = torch.cuda.max_memory_allocated() / 1e9 if _cuda else 0.0
            rows.append(dict(BATCH=bs, img_s=ips, VRAM_GB=gb,
                             phut_moi_epoch=len(I_TF) / ips / 60,
                             gio_cho_E=len(I_TF) / ips * EPOCHS / 3600))
            print(f'  BATCH {bs:>4}: {ips:6.0f} img/s | VRAM {gb:5.1f} GB | '
                  f'{rows[-1]["phut_moi_epoch"]:5.1f} ph/epoch | {rows[-1]["gio_cho_E"]:4.1f} h cho E={EPOCHS}')
            del dl, it, opt_
        except torch.cuda.OutOfMemoryError:
            print(f'  BATCH {bs:>4}: OOM'); break
    opt_ = None
    if _cuda: torch.cuda.empty_cache()
    return pd.DataFrame(rows)

PR = probe()
if len(PR):
    _best = PR.loc[PR.img_s.idxmax()]
    print(f'\nNhanh nhat: BATCH {int(_best.BATCH)} -> {_best.img_s:.0f} img/s, '
          f'{_best.VRAM_GB:.1f} GB, {_best.gio_cho_E:.1f} gio cho {EPOCHS} epoch')
    print(f'  ACCUM de giu EFF~240: {max(1, round(240/_best.BATCH))}')
    print(f'  BATCH hien tai o C0 la {BATCH} — sua roi CHAY LAI C0 neu muon doi.')
    print('\n  DOC: `img/s` gan nhu khong tang khi BATCH tang => IO-bound, lever la NW/giai ma.')
    PR.to_csv(os.path.join(RES38, 'batch_probe.csv'), index=False)
# tra lai trang thai sach cho C7
for m in (bb_ft, HEAD): m.zero_grad(set_to_none=True)

## C6 · Chấm điểm — verbatim `32` C2 / `35` C3 / `37` C6

In [ ]:
from sklearn.metrics import average_precision_score
_sid_va = ORDER.study_id.to_numpy()[VA]
_uid, _inv = np.unique(_sid_va, return_inverse=True)
_isF = (V.view_group.to_numpy()[VA] == 'frontal') & V.use.to_numpy()[VA]
_isL = (V.view_group.to_numpy()[VA] == 'lateral') & V.use.to_numpy()[VA]

def pool_study(p):
    out = np.zeros((len(_uid), p.shape[1]), 'float64'); ws = np.zeros(len(_uid))
    for mask, w in ((_isF, W_FRONTAL), (_isL, W_LATERAL)):
        if not mask.any(): continue
        cnt = np.bincount(_inv[mask], minlength=len(_uid))
        acc = np.stack([np.bincount(_inv[mask], weights=p[mask, j].astype('float64'),
                                    minlength=len(_uid)) for j in range(p.shape[1])], 1)
        has = cnt > 0
        out[has] += w * acc[has] / cnt[has, None]; ws[has] += w
    ok = ws > 0; out[ok] /= ws[ok, None]; return out, ok
_first = np.zeros(len(_uid), int); _first[_inv[::-1]] = np.arange(VA.sum())[::-1]
_Ys = YM[VA][_first]

def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, s = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-s, kind='stable'); yy, ss = y[r], s[r]
        tp, fp = np.cumsum(yy), np.cumsum(1 - yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp / (tp + fp), tp / n1
        o[j] = float((np.diff(np.r_[0.0, rc]) * pr).sum())
    return o
tiers = lambda ap: {t: float(np.nanmean(ap[i])) for t, i in TIER.items()}

def score(p):
    ap_i = ap_cols(YM[VA], p)
    ps, ok = pool_study(p)
    ap_s = np.array([average_precision_score(_Ys[ok][:, j], ps[ok][:, j]) for j in range(C)])
    return ap_i, ap_s
print(f'val {VA.sum():,} anh -> {len(_uid):,} study')

## C7 · Vòng lặp

Không early-stop, cosine trải đúng trên `EPOCHS`, lấy **epoch cuối**. Ghi đường cong val **theo
tier** mỗi epoch — đó là cách duy nhất trả lời "bao nhiêu epoch" bằng dữ liệu thay vì bằng `04b`.

Clip **từng arm** (`AMENDMENTS` ①): clip chung tính một chuẩn toàn cục rồi chia đều, tức ghép hai arm
lại với nhau. `ft` có thêm `86M` tham số backbone nên chuẩn của nó lớn hơn `lp` rất nhiều — clip
chung sẽ bóp `lp` theo.

In [ ]:
crit = ASL(gn=4, gp=0).to(dev)
P_ARM = {'lp': [p for p in HEAD['lp'].parameters() if p.requires_grad],
         'ft': [p for p in list(HEAD['ft'].parameters()) + list(bb_ft.parameters())
                if p.requires_grad]}
N_CLIP = {t: 0 for t in P_ARM}; N_STEP = 0

def make_opt():
    '''Backbone LR = LR dau doc / 10 (LP-FT). LambdaLR nhan CHUNG mot he so len moi group
       nen ti le 1/10 giu nguyen suot lich.'''
    hp = [p for t in ('lp', 'ft') for p in HEAD[t].parameters() if p.requires_grad]
    bp = [p for p in bb_ft.parameters() if p.requires_grad]
    sp = lambda ps, lr: [{'params': [p for p in ps if p.ndim > 1],  'lr': lr, 'weight_decay': WD},
                         {'params': [p for p in ps if p.ndim <= 1], 'lr': lr, 'weight_decay': 0.0}]
    return torch.optim.AdamW(sp(hp, LR_HEAD) + sp(bp, LR_HEAD * LR_MULT_BB))

@torch.no_grad()
def predict(idx, bs=64):
    bb_ft.eval(); HEAD.eval()
    P = {t: np.zeros((len(idx), C), 'float32') for t in ('lp', 'ft')}
    i = 0
    for b in tqdm(loader(idx, eval_tf, False, bs, nw=0), desc='val', leave=False):
        img = b['image'].to(dev, non_blocking=True); n = img.shape[0]
        P['lp'][i:i+n] = torch.sigmoid(HEAD['lp'](fwd(bb_lp, img, False))[0].float()).cpu().numpy()
        P['ft'][i:i+n] = torch.sigmoid(HEAD['ft'](fwd(bb_ft, img, False))[0].float()).cpu().numpy()
        i += n
    _bad = {t: int(np.isnan(v).sum()) for t, v in P.items() if np.isnan(v).any()}
    assert not _bad, f'predict cho ra NaN: {_bad}'
    return P

P_LOG = os.path.join(RES38, 'epoch_log.json')
LOG = json.load(open(P_LOG)) if os.path.exists(P_LOG) else {}
DL = loader(I_TF, train_tf, True, BATCH)
opt = make_opt()
steps = math.ceil(len(I_TF) / (BATCH * ACCUM)) * EPOCHS; warm = int(WARMUP * steps)
sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: s/max(warm,1) if s < warm else
                                        0.5*(1+math.cos(math.pi*(s-warm)/max(steps-warm,1))))
start = 0; _rp = os.path.join(CKPT, 'resume.pt')
if os.path.exists(_rp):
    ck = torch.load(_rp, map_location=dev, weights_only=False)
    if set(ck['head']) != set(HEAD.state_dict()):
        print('!! ckpt thuoc lan chay khac -> BO QUA, bat dau tu epoch 0')
    else:
        HEAD.load_state_dict(ck['head']); bb_ft.load_state_dict(ck['bb'])
        opt.load_state_dict(ck['opt']); sch.load_state_dict(ck['sch']); start = ck['ep'] + 1
        print(f'RESUME tu epoch {start+1}')

for ep in range(start, EPOCHS):
    bb_ft.train(); HEAD.train()
    t0, tot, nb = time.time(), 0.0, 0
    opt.zero_grad(set_to_none=True)
    for k, b in enumerate(tqdm(DL, desc=f'ep{ep+1}/{EPOCHS}', leave=False)):
        img = b['image'].to(dev, non_blocking=True); y = b['y'].to(dev)
        lg_lp, _ = HEAD['lp'](fwd(bb_lp, img, False))          # backbone dong cung
        lg_ft, _ = HEAD['ft'](fwd(bb_ft, img, True))           # backbone HOC
        for _t, _lg in (('lp', lg_lp), ('ft', lg_ft)):
            if not torch.isfinite(_lg).all():
                raise RuntimeError(f'ep{ep+1} buoc {k}: arm `{_t}` cho ra logit NaN/inf')
        loss = crit(lg_lp.float(), y) + crit(lg_ft.float(), y)
        (loss / ACCUM).backward(); tot += float(loss.detach()); nb += 1
        if (k + 1) % ACCUM == 0:
            for _t, _ps in P_ARM.items():                      # clip RIENG tung arm
                if float(torch.nn.utils.clip_grad_norm_(_ps, CLIP)) > CLIP: N_CLIP[_t] += 1
            N_STEP += 1
            opt.step(); sch.step(); opt.zero_grad(set_to_none=True)
    P = predict(I_VA)
    for t, p in P.items(): np.save(os.path.join(RES38, f'p_{t}_ep{ep+1}.npy'), p)
    _s = {t: score(p)[0] for t, p in P.items()}
    LOG[str(ep)] = {t: dict(mAP_anh=float(np.nanmean(a)), **tiers(a)) for t, a in _s.items()}
    # --- LAT CHON: day moi la cho duoc phep chon `E`. Val CHI de bao cao. ---
    #   a1 va a2 deu chon E bang cach nhin duong cong VAL — dung kieu ro ri ma `_runs.jsonl`
    #   dang dem. C7 truoc do khong cham lat chon nen quy tac "chon E tren lat cat 15%" chua
    #   bao gio duoc thi hanh. Chi dung `ap_cols` (muc ANH): `score()` gop study bang `_uid`
    #   cua VAL nen khong ap duoc cho lat chon.
    PS = predict(I_SL)
    LOG[str(ep)]['_sel'] = {t: float(np.nanmean(ap_cols(YM[SL], p))) for t, p in PS.items()}
    json.dump(LOG, open(P_LOG, 'w'))
    _sl = LOG[str(ep)]['_sel']
    print(f'   LAT CHON: lp {_sl["lp"]:.4f} | ft {_sl["ft"]:.4f} | ft-lp {_sl["ft"]-_sl["lp"]:+.4f}'
          f'   <- CHON `E` BANG COT NAY, khong bang val')
    torch.save(dict(head=HEAD.state_dict(), bb=bb_ft.state_dict(), opt=opt.state_dict(),
                    sch=sch.state_dict(), ep=ep), _rp)
    _ips = nb * BATCH / max(time.time() - t0, 1e-9)
    print(f'ep{ep+1:>2} loss {tot/max(nb,1):.4f} | lp {LOG[str(ep)]["lp"]["mAP_anh"]:.4f} '
          f'(tail {LOG[str(ep)]["lp"]["tail"]:.4f}) | ft {LOG[str(ep)]["ft"]["mAP_anh"]:.4f} '
          f'(tail {LOG[str(ep)]["ft"]["tail"]:.4f}) | ft-lp '
          f'{LOG[str(ep)]["ft"]["mAP_anh"]-LOG[str(ep)]["lp"]["mAP_anh"]:+.4f} | '
          f'{_ips:.0f} img/s | {(time.time()-t0)/60:.1f} ph')
    _f = {t: N_CLIP[t]/max(N_STEP,1) for t in P_ARM}
    if max(_f.values()) > 0.01:
        print('   clip kich hoat: ' + ' '.join(f'{t}:{v:.0%}' for t, v in _f.items() if v > 0.01))
print('xong')

## C8 · Kết quả + cổng + đường cong **theo tier**

In [ ]:
LOG = json.load(open(P_LOG)); EP = max(int(k) for k in LOG)
print(f'doc EPOCH CUOI = ep{EP+1} (khong early-stop)')
R = {}
for t in ('lp', 'ft'):
    ai, asx = score(np.load(os.path.join(RES38, f'p_{t}_ep{EP+1}.npy')))
    R[t] = dict(mAP_anh=float(ai.mean()), mAP_study=float(asx.mean()), **tiers(ai),
                **{f'ap_{l}': v for l, v in zip(LAB40, ai)})
T = pd.DataFrame(R).T
print('\n=== BANG CHINH (val, epoch cuoi) ===')
print(T[['mAP_anh', 'mAP_study', 'head', 'medium', 'tail']].to_string(float_format=lambda v: f'{v:.4f}'))

g = T.loc['ft', 'mAP_anh'] - T.loc['lp', 'mAP_anh']
print(f'\n=== CONG: ft - lp >= +0.03 ===')
print(f'  ft - lp = {g:+.4f}   {"QUA CONG" if g >= 0.03 else "DUOI CONG"}   (cohort cu: +0.101)')
for k in ('head', 'medium', 'tail'):
    print(f'    {k:<7} {T.loc["ft", k] - T.loc["lp", k]:+.4f}')

print('\n--- duong cong THEO TIER: con dang len khong? ---')
for t in ('lp', 'ft'):
    CU = pd.DataFrame({m: {int(k)+1: v[t][m] for k, v in LOG.items()}
                       for m in ('mAP_anh', 'head', 'medium', 'tail')})
    print(f'\n{t}:'); print(CU.to_string(float_format=lambda v: f'{v:.4f}'))
    if len(CU) >= 3:
        d = CU.iloc[-1] - CU.iloc[-3]
        print('  3 epoch cuoi: ' + ' | '.join(f'{m} {d[m]:+.4f}' for m in CU.columns))
print()
print('--- LAT CHON (muc anh) — CHON `E` BANG BANG NAY ---')
if all('_sel' in v for v in LOG.values()):
    SEL = pd.DataFrame({t: {int(k)+1: v['_sel'][t] for k, v in LOG.items()} for t in ('lp','ft')})
    SEL['ft_tru_lp'] = SEL.ft - SEL.lp
    print(SEL.to_string(float_format=lambda v: f'{v:+.4f}'))
    _be = int(SEL.ft.idxmax())
    print()
    print(f'  ft dinh tren LAT CHON o ep{_be}/{EPOCHS}.')
    print(f'  ep{_be} == {EPOCHS} => diem u xong TRUNG dinh; day la `E` dung.')
    print(f'  ep{_be} <  {EPOCHS} => `E` van dai; thu E={_be}.')
else:
    print('  (lan chay nay chua co cot lat chon — chay lai C7 ban moi)')
print()
print('DOC duong cong tier: `tail` con len ma `head` da phang => tang EPOCHS.')
print('     `tail` TUT som => backbone quen dac trung pretrain: ha LR_MULT_BB hoac mo 4 block cuoi.')
T.to_csv(os.path.join(RES38, 'main_table.csv'))
json.dump(dict(run=RUN_TAG, E=EPOCHS, batch=BATCH, accum=ACCUM, lr_head=LR_HEAD,
               lr_mult_bb=LR_MULT_BB, ft_lp=float(g), ft=float(T.loc['ft','mAP_anh']),
               lp=float(T.loc['lp','mAP_anh'])),
          open(P_RUNS, 'a')); open(P_RUNS, 'a').write('\n')
print(f'\nda ghi mot dong vao {P_RUNS} — moi lan cham val deu duoc dem.')

## C9 · Lưu backbone đã fine-tune cho `38b`

`38b` sẽ **đóng băng** backbone này rồi chạy lại toàn bộ arm EHR — lúc đó một forward dùng chung,
mỗi arm ~15% chi phí, và chạy được 2–3 seed.

In [ ]:
P_BB = os.path.join(RES38, 'backbone_ft.pt')
torch.save(dict(bb=bb_ft.state_dict(), head=HEAD['ft'].state_dict(),
                run=RUN_TAG, E=EPOCHS, ep=EP+1,
                ft_lp=float(T.loc['ft','mAP_anh'] - T.loc['lp','mAP_anh'])), P_BB)
print(f'-> {P_BB}  ({os.path.getsize(P_BB)/1e6:.0f} MB)')
print(f'   38b nap file nay, DONG BANG `bb`, roi gan cac arm EHR len tren.')

## C10 · Ghi `REPORT_STAGE4_NB38A.md`

In [ ]:
L = []; w = L.append
w('# REPORT · `nb38a` — đóng góp của fine-tune (chỉ ảnh)'); w('')
w(f'`{RUN_TAG}` · `asl40` · `E={EPOCHS}` · **epoch cuối** · BATCH {BATCH}×{ACCUM} · '
  f'LR đầu đọc `{LR_HEAD}`, backbone `{LR_HEAD*LR_MULT_BB}` · val {VA.sum():,} ảnh.'); w('')
w('| arm | mAP ảnh | study | head | medium | tail |'); w('|---|---|---|---|---|---|')
for t in ('lp', 'ft'):
    r = T.loc[t]
    w(f'| `{t}` | {r.mAP_anh:.4f} | {r.mAP_study:.4f} | {r["head"]:.4f} | {r["medium"]:.4f} | {r["tail"]:.4f} |')
w(''); w(f'**`ft − lp` = `{g:+.4f}`** (cổng `+0.03`; cohort cũ đo `+0.101` trên 13 nhãn).')
w(f'Theo tier: head `{T.loc["ft","head"]-T.loc["lp","head"]:+.4f}` · '
  f'medium `{T.loc["ft","medium"]-T.loc["lp","medium"]:+.4f}` · '
  f'tail `{T.loc["ft","tail"]-T.loc["lp","tail"]:+.4f}`.'); w('')
w('## Những gì kết quả này KHÔNG loại trừ'); w('')
w('- **EHR.** Không một arm EHR nào ở đây. `38b` đóng băng backbone này rồi mới gắn arm EHR — đó là '
  'phép thử **khó nhất** cho EHR: ảnh đã tối đa rồi, EHR còn thêm gì.')
w('- **Mức mở.** Chỉ thử full fine-tune với `LR_backbone = LR_head/10`. Mở một phần (4 block cuối) '
  'và adapter chưa thử.')
w(f'- **Ngân sách.** Chỉ `E={EPOCHS}`. Đường cong theo tier ở C8 cho biết còn lên hay không.')
w('- **Đích.** Nhãn CXR-LT do NLP trích từ **chính report của phim đó** ⇒ ảnh gần như đủ theo định '
  'nghĩa. Không con số nào ở đây nói gì về tiên lượng, tử vong, chuyển ICU, tái nhập viện.')
TXT = '\n'.join(L)
open(os.path.join(RES38, 'REPORT_STAGE4_NB38A.md'), 'w', encoding='utf-8').write(TXT)
print(TXT)